# 🔧 Phase 3 : Feature Engineering
## E-Commerce ML Pipeline — FACT_ACTIVITE

**Objectif :** Transformer et enrichir les données brutes en features exploitables par les modèles ML.

| Étape | Description |
|-------|-------------|
| 3.1 | Chargement du dataset ML (issu Phase 2) |
| 3.2 | Transformations logarithmiques (skewness) |
| 3.3 | Features temporelles avancées |
| 3.4 | Features client & produit |
| 3.5 | Features Social Media (lagged) |
| 3.6 | Encodage des variables catégorielles |
| 3.7 | Normalisation / Standardisation |
| 3.8 | Sélection de features (importance) |
| 3.9 | Sauvegarde du dataset final |

---
**Input :** `data/processed/dataset_ml.csv`  
**Output :** `data/processed/dataset_ml_features.csv`

## 📦 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')

# Style
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 30)

# Chemins
BASE_DIR   = Path(r'c:\4_ERP_BI\Semestre_2\E-commerce\ML_Engineering')
INPUT_PATH = BASE_DIR / 'data' / 'processed' / 'dataset_ml.csv'
OUTPUT_PATH= BASE_DIR / 'data' / 'processed' / 'dataset_ml_features.csv'
FIG_DIR    = BASE_DIR / 'reports'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Imports OK')
print(f'Input  : {INPUT_PATH}')
print(f'Output : {OUTPUT_PATH}')

---
## 📂 Étape 3.1 — Chargement du dataset ML

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f'Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'\nColonnes disponibles :')
print(df.columns.tolist())
print(f'\nAperçu :')
display(df.head(3))

In [ ]:
# Types et valeurs manquantes
info_df = pd.DataFrame({
    'Type': df.dtypes,
    'Null': df.isnull().sum(),
    'Null%': (df.isnull().sum() / len(df) * 100).round(1),
    'Unique': df.nunique()
})
print('=== TYPES & VALEURS MANQUANTES ===')
display(info_df)

---
## 📐 Étape 3.2 — Transformations Logarithmiques

> **Rappel EDA Phase 2 :** Toutes les features numériques ont une skewness > 10.  
> La transformation `log1p` (log(x+1)) normalise la distribution et améliore les performances des modèles.

In [ ]:
# Colonnes à transformer (uniquement celles avec valeurs >= 0)
log_cols = ['Montant_TTC', 'Montant_HT', 'Montant_TVA', 
            'Prix_unitaire', 'Quantite', 'Prix_concurrent',
            'Likes', 'Commentaires']
log_cols = [c for c in log_cols if c in df.columns]

# Vérifier les valeurs négatives
print('=== VÉRIFICATION VALEURS NÉGATIVES ===')
for col in log_cols:
    nb_neg = (df[col] < 0).sum()
    if nb_neg > 0:
        print(f'  ⚠️  {col} : {nb_neg} valeurs négatives → np.abs avant log')
    else:
        print(f'  ✅ {col} : OK (min={df[col].min():.2f})')

# Comparaison avant/après transformation
fig, axes = plt.subplots(2, len(log_cols), figsize=(18, 6))
fig.suptitle('Transformation log1p — Avant vs Après', fontsize=14, fontweight='bold')

for i, col in enumerate(log_cols):
    vals = df[col].clip(lower=0)  # sécurité
    log_vals = np.log1p(vals)
    
    axes[0, i].hist(vals, bins=30, color='#e74c3c', alpha=0.7, edgecolor='white')
    axes[0, i].set_title(f'{col}\nSkew={vals.skew():.1f}', fontsize=8)
    axes[0, i].set_xlabel('Original', fontsize=7)
    
    axes[1, i].hist(log_vals, bins=30, color='#2ecc71', alpha=0.7, edgecolor='white')
    axes[1, i].set_title(f'log1p({col})\nSkew={log_vals.skew():.1f}', fontsize=8)
    axes[1, i].set_xlabel('log1p', fontsize=7)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_log_transform.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Visualisation sauvegardée')

In [ ]:
# Appliquer log1p et créer les nouvelles colonnes
df_feat = df.copy()

for col in log_cols:
    df_feat[f'log_{col}'] = np.log1p(df_feat[col].clip(lower=0))

print(f'✅ {len(log_cols)} features log1p créées :')
new_log_cols = [f'log_{c}' for c in log_cols]
print(new_log_cols)

# Résumé skewness
skew_before = df[log_cols].skew().round(2)
skew_after  = df_feat[new_log_cols].skew().round(2)
skew_df = pd.DataFrame({'Feature': log_cols, 'Skew_avant': skew_before.values, 
                        'Skew_après': skew_after.values})
skew_df['Amélioration'] = (skew_df['Skew_avant'].abs() - skew_df['Skew_après'].abs()).round(2)
display(skew_df)

---
## 📅 Étape 3.3 — Features Temporelles Avancées

Extraction depuis `FK_Date` (format YYYYMMDD) et création de features cycliques.

In [ ]:
# Conversion FK_Date → datetime
if 'FK_Date' in df_feat.columns:
    df_feat['Date'] = pd.to_datetime(df_feat['FK_Date'].astype(str), format='%Y%m%d', errors='coerce')
    nb_valid = df_feat['Date'].notna().sum()
    print(f'✅ Dates converties : {nb_valid}/{len(df_feat)} valides')

    # === Features de base (si pas déjà présentes depuis Phase 2) ===
    if 'Annee' not in df_feat.columns:
        df_feat['Annee']     = df_feat['Date'].dt.year
    if 'Mois' not in df_feat.columns:
        df_feat['Mois']      = df_feat['Date'].dt.month
    
    df_feat['Jour']          = df_feat['Date'].dt.day
    df_feat['Jour_semaine']  = df_feat['Date'].dt.dayofweek        # 0=Lun, 6=Dim
    df_feat['Jour_an']       = df_feat['Date'].dt.dayofyear
    df_feat['Semaine']       = df_feat['Date'].dt.isocalendar().week.astype(int)
    
    # === Indicateurs binaires ===
    if 'Est_WeekEnd' not in df_feat.columns:
        df_feat['Est_WeekEnd'] = df_feat['Jour_semaine'].isin([5, 6]).astype(int)
    
    df_feat['Est_Debut_Mois']  = (df_feat['Jour'] <= 10).astype(int)
    df_feat['Est_Fin_Mois']    = (df_feat['Jour'] >= 20).astype(int)
    df_feat['Est_Q4']          = df_feat['Mois'].isin([10, 11, 12]).astype(int)
    df_feat['Est_Ete']         = df_feat['Mois'].isin([6, 7, 8]).astype(int)

    # === Features cycliques (sin/cos) — pour les modèles qui comprennent la circularité ===
    df_feat['Mois_sin']        = np.sin(2 * np.pi * df_feat['Mois'] / 12)
    df_feat['Mois_cos']        = np.cos(2 * np.pi * df_feat['Mois'] / 12)
    df_feat['Jour_sem_sin']    = np.sin(2 * np.pi * df_feat['Jour_semaine'] / 7)
    df_feat['Jour_sem_cos']    = np.cos(2 * np.pi * df_feat['Jour_semaine'] / 7)

    print(f'\n✅ Features temporelles créées :')
    temp_cols = ['Jour', 'Jour_semaine', 'Semaine', 'Jour_an',
                 'Est_Debut_Mois', 'Est_Fin_Mois', 'Est_Q4', 'Est_Ete',
                 'Mois_sin', 'Mois_cos', 'Jour_sem_sin', 'Jour_sem_cos']
    print(temp_cols)
    display(df_feat[['Date', 'Annee', 'Mois', 'Jour'] + temp_cols[:6]].head(5))
else:
    print('⚠️ FK_Date absent du dataset')

In [ ]:
# Visualisation des features cycliques
fig, axes = plt.subplots(1, 2, figsize=(12, 4), subplot_kw=dict(projection='polar'))
fig.suptitle('Features Cycliques — Encodage Sin/Cos', fontsize=12, fontweight='bold')

# Mois
theta_mois = np.linspace(0, 2 * np.pi, 12, endpoint=False)
r_mois = df_feat.groupby('Mois')['log_Montant_TTC'].mean().reindex(range(1, 13), fill_value=0).values if 'log_Montant_TTC' in df_feat.columns else np.ones(12)
r_mois = np.clip(r_mois, 0, None)
axes[0].bar(theta_mois, r_mois, width=2*np.pi/12, alpha=0.7, color='#3498db', edgecolor='white')
axes[0].set_title('CA moyen par Mois\n(log scale)', fontsize=10)
axes[0].set_xticks(theta_mois)
axes[0].set_xticklabels(['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc'], fontsize=7)

# Jour de la semaine
theta_sem = np.linspace(0, 2 * np.pi, 7, endpoint=False)
r_sem = df_feat.groupby('Jour_semaine')['log_Montant_TTC'].mean().reindex(range(7), fill_value=0).values if 'log_Montant_TTC' in df_feat.columns else np.ones(7)
r_sem = np.clip(r_sem, 0, None)
axes[1].bar(theta_sem, r_sem, width=2*np.pi/7, alpha=0.7, color='#e74c3c', edgecolor='white')
axes[1].set_title('CA moyen par Jour/Semaine\n(log scale)', fontsize=10)
axes[1].set_xticks(theta_sem)
axes[1].set_xticklabels(['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'], fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_cyclical_features.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 👤 Étape 3.4 — Features Client & Produit (Agrégées)

In [ ]:
if 'FK_Client' in df_feat.columns and 'Montant_TTC' in df_feat.columns:
    # === Agrégations par client ===
    client_stats = df_feat[df_feat['FK_Client'] > 0].groupby('FK_Client').agg(
        client_nb_achats    = ('Montant_TTC', 'count'),
        client_ca_total     = ('Montant_TTC', 'sum'),
        client_ca_moyen     = ('Montant_TTC', 'mean'),
        client_ca_max       = ('Montant_TTC', 'max'),
        client_ca_std       = ('Montant_TTC', 'std')
    ).reset_index()
    client_stats['client_ca_std'] = client_stats['client_ca_std'].fillna(0)
    
    # Merge
    df_feat = df_feat.merge(client_stats, on='FK_Client', how='left')
    
    # Remplir les clients sans historique (FK_Client = -1)
    for col in ['client_nb_achats', 'client_ca_total', 'client_ca_moyen', 'client_ca_max', 'client_ca_std']:
        df_feat[col] = df_feat[col].fillna(0)
    
    # Segment client (RFM simplifié)
    bins = [0, 1, 3, 10, float('inf')]
    labels = [0, 1, 2, 3]  # Occasionnel, Régulier, Fidèle, VIP
    df_feat['Segment_Client'] = pd.cut(df_feat['client_nb_achats'], bins=bins, labels=labels, right=True)
    df_feat['Segment_Client'] = df_feat['Segment_Client'].astype(float).fillna(0).astype(int)
    
    print('=== FEATURES CLIENT ===')
    print(f'✅ 5 agrégations + 1 segment créés')
    display(df_feat[['FK_Client', 'client_nb_achats', 'client_ca_moyen', 'Segment_Client']].head(5))
else:
    print('⚠️ FK_Client ou Montant_TTC absent')

In [ ]:
if 'FK_Produit' in df_feat.columns and 'Montant_TTC' in df_feat.columns:
    # Garder seulement les vrais FK_Produit (> 0)
    mask_prod = df_feat['FK_Produit'] > 0
    
    if mask_prod.sum() > 0:
        prod_stats = df_feat[mask_prod].groupby('FK_Produit').agg(
            prod_nb_ventes  = ('Montant_TTC', 'count'),
            prod_ca_total   = ('Montant_TTC', 'sum'),
            prod_ca_moyen   = ('Montant_TTC', 'mean'),
            prod_qte_moy    = ('Quantite', 'mean') if 'Quantite' in df_feat.columns else ('Montant_TTC', 'count')
        ).reset_index()
        
        df_feat = df_feat.merge(prod_stats, on='FK_Produit', how='left')
        for col in prod_stats.columns[1:]:
            df_feat[col] = df_feat[col].fillna(0)
        print(f'✅ Features produit créées pour {mask_prod.sum()} lignes avec produit identifié')
    else:
        # Tous FK_Produit = -1
        df_feat['prod_nb_ventes'] = 0
        df_feat['prod_ca_total']  = 0
        df_feat['prod_ca_moyen']  = 0
        print('ℹ️ FK_Produit = -1 pour toutes les lignes (données agrégées)')
else:
    print('⚠️ FK_Produit absent')

---
## 📱 Étape 3.5 — Features Social Media (Lagged)

> **Hypothèse :** L'impact des réseaux sociaux sur les ventes peut être décalé de quelques jours/semaines.  
> On crée des features avec décalage temporel pour capturer cet effet retardé.

In [ ]:
if 'Likes' in df_feat.columns and 'Date' in df_feat.columns:
    # Agréger les Likes par semaine
    df_feat_sorted = df_feat.sort_values('Date').copy()
    df_feat_sorted['Semaine_date'] = df_feat_sorted['Date'].dt.to_period('W')
    
    # Agrégation hebdomadaire des Likes
    weekly_likes = df_feat_sorted.groupby('Semaine_date')['Likes'].sum().reset_index()
    weekly_likes.columns = ['Semaine_date', 'Likes_semaine']
    
    # Lag 1, 2, 4 semaines
    weekly_likes['Likes_lag1w']  = weekly_likes['Likes_semaine'].shift(1)
    weekly_likes['Likes_lag2w']  = weekly_likes['Likes_semaine'].shift(2)
    weekly_likes['Likes_lag4w']  = weekly_likes['Likes_semaine'].shift(4)
    weekly_likes['Likes_roll4w'] = weekly_likes['Likes_semaine'].rolling(4, min_periods=1).mean()
    
    # Merge back
    df_feat_sorted = df_feat_sorted.merge(weekly_likes, on='Semaine_date', how='left')
    
    for col in ['Likes_lag1w', 'Likes_lag2w', 'Likes_lag4w', 'Likes_roll4w']:
        df_feat_sorted[col] = df_feat_sorted[col].fillna(0)
    
    df_feat = df_feat_sorted.copy()
    
    print('=== FEATURES SOCIAL MEDIA LAGGED ===')
    print('✅ Créées :') 
    print('  - Likes_lag1w  : Likes semaine-1')
    print('  - Likes_lag2w  : Likes semaine-2')
    print('  - Likes_lag4w  : Likes semaine-4')
    print('  - Likes_roll4w : Moyenne mobile 4 semaines')
    
    # Corrélation avec CA
    lag_cols = ['Likes', 'Likes_lag1w', 'Likes_lag2w', 'Likes_lag4w', 'Likes_roll4w']
    lag_cols = [c for c in lag_cols if c in df_feat.columns]
    
    if 'Montant_TTC' in df_feat.columns:
        corr_lag = df_feat[lag_cols + ['Montant_TTC']].corr()['Montant_TTC'].drop('Montant_TTC').round(3)
        print('\n=== CORRÉLATIONS AVEC Montant_TTC ===')
        for col, val in corr_lag.items():
            icon = '🔴' if abs(val) > 0.1 else '⚪'
            print(f'  {icon} {col:20s} : {val:+.3f}')
else:
    print('ℹ️ Likes non disponibles ou Date absente — features lagged ignorées')

---
## 💰 Étape 3.5b — Feature : Impact Remise

In [ ]:
if 'Remise' in df_feat.columns:
    # Tranche de remise
    df_feat['Tranche_Remise'] = pd.cut(
        df_feat['Remise'],
        bins=[-0.1, 0, 10, 25, 50, 100],
        labels=[0, 1, 2, 3, 4],  # 0=Aucune, 1=Faible, 2=Moyenne, 3=Forte, 4=Exceptionnelle
        right=True
    ).astype(float).fillna(0).astype(int)
    
    # Feature interaction : Remise × Prix
    if 'Prix_unitaire' in df_feat.columns:
        df_feat['Remise_Montant'] = df_feat['Prix_unitaire'] * df_feat['Remise'] / 100
        df_feat['log_Remise_Montant'] = np.log1p(df_feat['Remise_Montant'].clip(lower=0))
    
    print('=== DISTRIBUTION DES TRANCHES DE REMISE ===')
    labels_map = {0: 'Aucune (0%)', 1: 'Faible (1-10%)', 2: 'Moyenne (10-25%)', 
                  3: 'Forte (25-50%)', 4: 'Exceptionnelle (>50%)'}
    dist = df_feat['Tranche_Remise'].value_counts().sort_index()
    for k, v in dist.items():
        print(f'  {labels_map.get(k, k):30s} : {v:4d} ({v/len(df_feat)*100:.1f}%)')
else:
    print('ℹ️ Colonne Remise absente')

---
## 🔢 Étape 3.6 — Encodage des Variables Catégorielles

In [ ]:
# Colonnes catégorielles texte
cat_cols = df_feat.select_dtypes(include=['object']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['Date', 'Semaine_date']]

print(f'Colonnes catégorielles : {cat_cols}')

encoders = {}
for col in cat_cols:
    nb_unique = df_feat[col].nunique()
    print(f'  {col:20s} : {nb_unique} valeurs uniques', end='')
    
    if nb_unique <= 2:
        # Binaire
        df_feat[f'{col}_enc'] = LabelEncoder().fit_transform(df_feat[col].astype(str))
        print(' → Label Encoding')
    elif nb_unique <= 20:
        # Ordinal / Label
        le = LabelEncoder()
        df_feat[f'{col}_enc'] = le.fit_transform(df_feat[col].astype(str))
        encoders[col] = le
        print(' → Label Encoding')
    else:
        # Haute cardinalité → Frequency Encoding
        freq = df_feat[col].value_counts(normalize=True)
        df_feat[f'{col}_freq'] = df_feat[col].map(freq)
        print(' → Frequency Encoding')

# Colonnes FK (déjà numériques mais contiennent -1 → à remplacer par 0)
fk_cols = [c for c in df_feat.columns if c.startswith('FK_') and c != 'FK_Date']
for col in fk_cols:
    df_feat[col] = df_feat[col].replace(-1, 0)

print(f'\n✅ Encodage terminé. Shape : {df_feat.shape}')

---
## ⚖️ Étape 3.7 — Normalisation / Standardisation

In [ ]:
# Features numériques à scaler (hors target et IDs)
TARGET = 'log_Montant_TTC' if 'log_Montant_TTC' in df_feat.columns else 'Montant_TTC'
EXCLUDE = ['FK_Date', 'Date', 'Semaine_date', TARGET, 'Montant_TTC', 'Montant_HT', 'Montant_TVA']

scale_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
scale_cols = [c for c in scale_cols if c not in EXCLUDE and not c.startswith('Est_') 
              and not c.startswith('A_') and 'Segment' not in c and 'Tranche' not in c]

# Filtrer les colonnes avec variance > 0
scale_cols = [c for c in scale_cols if df_feat[c].std() > 0]

print(f'Colonnes à standardiser ({len(scale_cols)}) :')
print(scale_cols)

# StandardScaler (mean=0, std=1) — recommandé pour régression/SVM
scaler_std = StandardScaler()
X_scaled = scaler_std.fit_transform(df_feat[scale_cols])
df_scaled = pd.DataFrame(X_scaled, columns=[f'{c}_std' for c in scale_cols], index=df_feat.index)

# Ajouter au dataframe principal (versions standardisées)
df_feat = pd.concat([df_feat, df_scaled], axis=1)

print(f'\n✅ StandardScaler appliqué → {len(scale_cols)} colonnes _std créées')
print(f'   Dataset final : {df_feat.shape[0]} lignes × {df_feat.shape[1]} colonnes')

---
## 🎯 Étape 3.8 — Sélection de Features (Importance)

Identifier les features les plus pertinentes pour prédire `log_Montant_TTC`.

In [ ]:
# Préparer X et y
TARGET = 'log_Montant_TTC' if 'log_Montant_TTC' in df_feat.columns else 'Montant_TTC'

# Sélectionner uniquement les colonnes numériques et valides
feature_candidates = [
    # Temporelles
    'Annee', 'Mois', 'Jour', 'Jour_semaine', 'Semaine', 
    'Est_WeekEnd', 'Est_Q4', 'Est_Ete', 'Est_Debut_Mois', 'Est_Fin_Mois',
    'Mois_sin', 'Mois_cos', 'Jour_sem_sin', 'Jour_sem_cos',
    # Produit/Client
    'FK_Client', 'FK_Produit', 'FK_Geographie', 'FK_Fournisseur',
    'client_nb_achats', 'client_ca_moyen', 'Segment_Client',
    # Numeriques log
    'log_Prix_unitaire', 'log_Quantite', 'log_Prix_concurrent',
    'log_Likes', 'log_Commentaires',
    # Remise
    'Remise', 'A_Remise', 'Tranche_Remise',
    # Social
    'A_Social', 'Likes', 'Commentaires',
    # Lagged
    'Likes_lag1w', 'Likes_lag2w', 'Likes_lag4w', 'Likes_roll4w',
]

# Garder uniquement les colonnes présentes et numériques
feature_candidates = [c for c in feature_candidates 
                      if c in df_feat.columns and c != TARGET
                      and df_feat[c].dtype in [np.float64, np.int64, np.float32, np.int32]]

X = df_feat[feature_candidates].fillna(0)
y = df_feat[TARGET].fillna(0)

print(f'Features candidates : {len(feature_candidates)}')
print(f'Target              : {TARGET}')
print(f'Échantillons        : {len(X)}')

In [ ]:
# Random Forest Feature Importance
rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.DataFrame({
    'Feature': feature_candidates,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

# Top 15
top15 = importances.head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if imp > 0.05 else '#3498db' if imp > 0.02 else '#95a5a6' 
          for imp in top15['Importance']]
bars = ax.barh(top15['Feature'][::-1], top15['Importance'][::-1], color=colors[::-1], edgecolor='white')

ax.axvline(x=0.02, color='#f39c12', linestyle='--', alpha=0.7, label='Seuil 2%')
ax.set_title('Top 15 Features — Random Forest Importance\n(Target: log_Montant_TTC)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance (Gini)', fontsize=11)
ax.legend(fontsize=10)

for bar, val in zip(bars, top15['Importance'][::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, 
            f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== TOP 15 FEATURES ===')
display(top15.reset_index(drop=True))

In [ ]:
# Mutual Information (non-linéaire)
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_df = pd.DataFrame({
    'Feature': feature_candidates,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

print('=== MUTUAL INFORMATION — TOP 10 ===')
display(mi_df.head(10).reset_index(drop=True))

# Sélection finale : features importantes selon RF ou MI
threshold_rf = 0.01
threshold_mi = 0.005

selected_by_rf = set(importances[importances['Importance'] >= threshold_rf]['Feature'].tolist())
selected_by_mi = set(mi_df[mi_df['MI_Score'] >= threshold_mi]['Feature'].tolist())
SELECTED_FEATURES = list(selected_by_rf | selected_by_mi)

print(f'\n✅ Features sélectionnées (RF≥{threshold_rf} OU MI≥{threshold_mi}) : {len(SELECTED_FEATURES)}')
print(sorted(SELECTED_FEATURES))

---
## 🔥 Matrice de Corrélation — Features Sélectionnées

In [ ]:
# Matrice de corrélation des features sélectionnées + target
plot_cols = SELECTED_FEATURES + [TARGET]
plot_cols = [c for c in plot_cols if c in df_feat.columns]

corr = df_feat[plot_cols].corr()

fig, ax = plt.subplots(figsize=(max(10, len(plot_cols)), max(8, len(plot_cols)-2)))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, linewidths=0.5, annot_kws={'size': 8}, vmin=-1, vmax=1)
ax.set_title('Matrice de Corrélation — Features Sélectionnées', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_corr_final_features.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Matrice sauvegardée')

---
## 💾 Étape 3.9 — Sauvegarde du Dataset Final

In [ ]:
# Colonnes finales à conserver pour le modeling
KEEP_COLS = SELECTED_FEATURES + [TARGET]

# Ajouter quelques colonnes utiles pour le contexte
CONTEXT_COLS = ['FK_Client', 'FK_Date', 'Annee', 'Mois', 
                'Montant_TTC', 'Montant_HT']
CONTEXT_COLS = [c for c in CONTEXT_COLS if c in df_feat.columns and c not in KEEP_COLS]

FINAL_COLS = list(dict.fromkeys(CONTEXT_COLS + KEEP_COLS))  # dédoublonnage
FINAL_COLS = [c for c in FINAL_COLS if c in df_feat.columns]

df_final = df_feat[FINAL_COLS].copy()

# Supprimer les infinis et NaN résiduels
df_final.replace([np.inf, -np.inf], np.nan, inplace=True)
nb_nan_avant = df_final.isnull().sum().sum()
df_final.fillna(0, inplace=True)
print(f'ℹ️ {nb_nan_avant} NaN remplacés par 0')

# Sauvegarde
df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print(f'\n✅ Dataset ML Features sauvegardé :')
print(f'   Chemin : {OUTPUT_PATH}')
print(f'   Shape  : {df_final.shape[0]} lignes × {df_final.shape[1]} colonnes')
print(f'\nColonnes finales :')
print(FINAL_COLS)

In [ ]:
# 📋 RAPPORT RÉCAPITULATIF PHASE 3
print('=' * 60)
print('📋  RAPPORT — PHASE 3 : FEATURE ENGINEERING')
print('=' * 60)

print(f"\n📂 Input  : dataset_ml.csv ({len(df)} lignes × {len(df.columns)} cols)")
print(f"📂 Output : dataset_ml_features.csv ({df_final.shape[0]} lignes × {df_final.shape[1]} cols)")

print(f"\n🔧 TRANSFORMATIONS APPLIQUÉES :")
print(f"   ✅ Log1p sur {len(log_cols)} features numériques (réduction skewness)")
print(f"   ✅ Features temporelles : Jour, Semaine, cycliques Sin/Cos, indicateurs binaires")
print(f"   ✅ Features client : agrégations CA, nombre achats, segmentation")
print(f"   ✅ Features Social Media lagged : 1w, 2w, 4w, rolling 4w")
print(f"   ✅ Encodage catégoriel (Label/Frequency)")
print(f"   ✅ Standardisation (StandardScaler)")
print(f"   ✅ Sélection : {len(SELECTED_FEATURES)} features retenues (RF + MI)")

print(f"\n🎯 TARGET : {TARGET}")
print(f"   Distribution : mean={df_final[TARGET].mean():.3f}, std={df_final[TARGET].std():.3f}")

print(f"\n➡️ PROCHAINE ÉTAPE : Phase 4 — Modélisation ML")
print(f"   • Régression (prédiction CA)    → RandomForest, XGBoost, LightGBM")
print(f"   • Classification (segment CA)   → Seuils Bas/Moyen/Haut")
print(f"   • Clustering (profils clients)  → K-Means, DBSCAN")
print(f"   • Série temporelle (tendances)  → Prophet, ARIMA")
print('=' * 60)